# QUARTS full GPU experiment
This notebook is a reproducible front end to the frozen experiment scripts. Select the **research-gpu** kernel. Long stages are resumable: completed `metrics.json` artifacts are skipped. The live command-line run uses the same interpreter and scripts.

In [ ]:
from pathlib import Path
import os, sys, torch
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
os.chdir(ROOT)
print('workspace:', ROOT)
print('python:', sys.executable)
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
assert torch.cuda.is_available()

## Validation
Run the automated checks before launching or resuming experiments.

In [ ]:
%run -i scripts/run_smoke.py
!{sys.executable} -m pytest -q

## Confirmatory reference matrix
Do not execute this cell while the current background matrix is active. It is safe to execute later because it skips completed runs.

In [ ]:
!{sys.executable} scripts/run_matrix.py --config configs/full_experiment.yaml --datasets metr_la pems_bay --output artifacts/runs/confirmatory

## Remaining preregistered stages
This executes the resumable continuation: protocol-correction rerun, PEMS-BAY matrix, corruption robustness, zero-shot transfer, quantum noise, qubit/depth ablations, official STAEformer baseline, statistics, and figures.

In [ ]:
!{sys.executable} scripts/run_remaining_pipeline.py

## Inspect structured outputs

In [ ]:
import pandas as pd
metrics = ROOT / 'artifacts/analysis/run_metrics.csv'
display(pd.read_csv(metrics).groupby(['dataset','model'])[['MAE','RMSE','MAPE','WAPE']].agg(['mean','std'])) if metrics.exists() else print('Analysis is not complete yet.')